In [1]:
import random
import numpy as np

class TicTacToe:

    def __init__(self):
        self.reset()

    def reset(self):
        self.board = [' '] * 9
        self.done = False
        return self.get_state()

    def get_state(self):
        return ''.join(self.board)

    def available_actions(self):
        return [i for i in range(9) if self.board[i] == ' ']

    def winner(self, player):
        wins = [
            (0,1,2),(3,4,5),(6,7,8),
            (0,3,6),(1,4,7),(2,5,8),
            (0,4,8),(2,4,6)
        ]

        for w in wins:
            if all(self.board[i] == player for i in w):
                return True
        return False

    def step(self, action, player):

        self.board[action] = player

        if self.winner(player):
            self.done = True
            return self.get_state(), 1, True

        if len(self.available_actions()) == 0:
            self.done = True
            return self.get_state(), 0, True

        return self.get_state(), 0, False

    def display(self):

        for i in range(0,9,3):
            print(self.board[i:i+3])
        print()

In [2]:
class QLearningAgent:

    def __init__(self):

        self.q_table = {}

        self.alpha = 0.1
        self.gamma = 0.9
        self.epsilon = 0.2

    def get_q(self, state):

        if state not in self.q_table:
            self.q_table[state] = np.zeros(9)

        return self.q_table[state]

    def choose_action(self, state, actions):

        if random.random() < self.epsilon:
            return random.choice(actions)

        q = self.get_q(state)

        values = [q[a] for a in actions]

        return actions[np.argmax(values)]

    def update(self, state, action, reward, next_state, next_actions):

        q = self.get_q(state)

        next_q = self.get_q(next_state)

        if len(next_actions) == 0:
            target = reward
        else:
            target = reward + self.gamma * max(next_q[a] for a in next_actions)

        q[action] += self.alpha * (target - q[action])

In [3]:
env = TicTacToe()

agent = QLearningAgent()

episodes = 50000

for episode in range(episodes):

    state = env.reset()

    while True:

        actions = env.available_actions()

        action = agent.choose_action(state, actions)

        next_state, reward, done = env.step(action, 'X')

        if done:

            if reward == 1:
                final_reward = 1
            else:
                final_reward = 0

            agent.update(
                state,
                action,
                final_reward,
                next_state,
                []
            )

            break

        opponent_actions = env.available_actions()

        opp_action = random.choice(opponent_actions)

        next_state2, reward2, done = env.step(opp_action, 'O')

        if done:

            if reward2 == 1:
                final_reward = -1
            else:
                final_reward = 0

            agent.update(
                state,
                action,
                final_reward,
                next_state2,
                []
            )

            break

        next_actions = env.available_actions()

        agent.update(
            state,
            action,
            0,
            next_state2,
            next_actions
        )

        state = next_state2

print("Training Complete!")

Training Complete!


In [4]:
env.reset()

while not env.done:

    env.display()

    actions = env.available_actions()

    state = env.get_state()

    q = agent.get_q(state)

    best_action = actions[np.argmax([q[a] for a in actions])]

    env.step(best_action, 'X')

    if env.done:
        break

    env.display()

    move = int(input("Enter position (0-8): "))

    while move not in env.available_actions():
        move = int(input("Invalid move. Try again: "))

    env.step(move, 'O')

env.display()

if env.winner('X'):
    print("RL Agent Wins!")

elif env.winner('O'):
    print("You Win!")

else:
    print("Draw!")

[' ', ' ', ' ']
[' ', ' ', ' ']
[' ', ' ', ' ']

['X', ' ', ' ']
[' ', ' ', ' ']
[' ', ' ', ' ']

['X', ' ', ' ']
[' ', 'O', ' ']
[' ', ' ', ' ']

['X', 'X', ' ']
[' ', 'O', ' ']
[' ', ' ', ' ']

['X', 'X', 'O']
[' ', 'O', ' ']
[' ', ' ', ' ']

['X', 'X', 'O']
[' ', 'O', ' ']
['X', ' ', ' ']

['X', 'X', 'O']
['O', 'O', ' ']
['X', ' ', ' ']

['X', 'X', 'O']
['O', 'O', 'X']
['X', ' ', ' ']

['X', 'X', 'O']
['O', 'O', 'X']
['X', 'O', ' ']

['X', 'X', 'O']
['O', 'O', 'X']
['X', 'O', 'X']

Draw!
